# Influenza in an English boarding school, 1978

**C1 Research Computing, Lecture 1 demonstration.** A first notebook for the running project of the course.

In January 1978 an influenza A/H1N1 outbreak spread through a boarding school in the north of England. Of the 763 boys, 512 were confined to bed at some point. The school recorded, every morning for 14 days, the number of boys in bed.

*Source:* Anonymous, "Influenza in a boarding school", *British Medical Journal*, 4 March 1978, p. 587. Values as in the CRAN package `outbreaks` (dataset `influenza_england_1978_school`).

In [ ]:
import sys
print(sys.executable)   # which interpreter (and therefore which environment) runs this notebook
print(sys.version.split()[0])
print("inside a venv:", sys.prefix != sys.base_prefix)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Day 0 = 22 January 1978, day 13 = 4 February 1978
day    = np.arange(14)
in_bed = np.array([3, 8, 26, 76, 225, 298, 258, 233, 189, 128, 68, 29, 14, 4])
N = 763   # boys in the school

for d, b in zip(day, in_bed):
    print(f"day {d:2d}: {b:3d} in bed")

## The SIR model

The school is split into susceptible $S$, infected $I$ and recovered $R$ boys, with $S + I + R = N$:

$$\frac{dS}{dt} = -\beta \frac{S I}{N}, \qquad \frac{dI}{dt} = \beta \frac{S I}{N} - \gamma I, \qquad \frac{dR}{dt} = \gamma I .$$

$\beta$ is the transmission rate (per day), $1/\gamma$ the mean time spent infected (days), and $R_0 = \beta/\gamma$ the basic reproduction number. The number of boys in bed is compared with $I(t)$.

The equations are integrated below with the simplest possible scheme, explicit Euler: $y_{n+1} = y_n + \Delta t\, f(y_n)$.

In [ ]:
def sir_euler(beta, gamma, I0, N, t_max, dt=0.01):
    """Integrate the SIR equations with explicit Euler; return t, S, I, R."""
    n = int(round(t_max / dt))
    t = np.linspace(0.0, n * dt, n + 1)
    S, I, R = np.empty(n + 1), np.empty(n + 1), np.empty(n + 1)
    S[0], I[0], R[0] = N - I0, I0, 0.0
    for k in range(n):
        new_inf = beta * S[k] * I[k] / N * dt
        new_rec = gamma * I[k] * dt
        S[k + 1] = S[k] - new_inf
        I[k + 1] = I[k] + new_inf - new_rec
        R[k + 1] = R[k] + new_rec
    return t, S, I, R

In [ ]:
beta, gamma = 1.7, 0.45     # per day; coarse grid search against the data (a proper fit comes later)
t, S, I, R = sir_euler(beta, gamma, I0=3, N=N, t_max=13)

print(f"R0 = beta/gamma = {beta / gamma:.2f}")
print(f"mean infectious period = {1 / gamma:.1f} days")
print(f"model peak: {I.max():.0f} boys on day {t[I.argmax()]:.1f}")
print(f"S + I + R at the end = {S[-1] + I[-1] + R[-1]:.1f}  (should be {N})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(day, in_bed, "o", color="black", label="boys in bed (BMJ 1978)")
ax.plot(t, I, "-", label=rf"SIR model, $\beta$={beta}, $\gamma$={gamma}")
ax.set_xlabel("day (0 = 22 January 1978)")
ax.set_ylabel("number of boys")
ax.set_title("Influenza in a boarding school")
ax.legend()
plt.show()

## Next steps

The model reproduces the shape of the outbreak: a rise over four days, a peak near day 5 and a decay over a week. The parameters come from a coarse grid search (steps of 0.1 in beta and 0.05 in gamma, least squares); fitting them properly, testing the integrator, and replacing explicit Euler by a better scheme all come later in the course.

This notebook is where the course's running project, the **`attractor`** package, starts. Over the term the function `sir_euler` will move out of the notebook into an installable, version-controlled, tested Python package, which will then grow to cover Lotka-Volterra predator-prey dynamics, a genetic toggle switch and Gray-Scott reaction-diffusion patterns.